**Aim:**

To use the GPT-2 language model with Hugging Face Transformers to generate text continuations, compare different decoding strategies and fine-tune the last Transformer block for generating short study tips.

**Software and Libraries Used**:
1. Python
2. Google Colab
3. PyTorch
4. Hugging Face Transformers
5. Matplotlib
Pre-trained Model: openai-community/gpt2

2. **Theory**

GPT-2 is a Generative Pre-trained Transformer model that generates text by predicting the next token based on previously generated tokens.

It works through the following process:
1. Input Prompt – User enters a starting sentence.
2. Tokenization – Text is converted into tokens.
3. GPT-2 Model – Predicts the next token.
4. Decoding – Selects the next token using a decoding strategy.
5. Text Generation – Continues generating text.
Decoding strategies:
- Greedy decoding: Selects the highest-probability token.
- Sampling: Selects tokens based on probability.
- Temperature: Controls randomness in generated text.

In [1]:

%pip install -q "transformers>=4.51,<5"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [6]:



import torch
import matplotlib.pyplot as plt

from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)


SEED = 42
torch.manual_seed(SEED)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


out = Path("lab_outputs")
out.mkdir(exist_ok=True)

print("Device:", device)


model_id = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id
).to(device)

model.eval()

print("GPT-2 loaded successfully!")
print("Model:", model_id)


Device: cpu
GPT-2 loaded successfully!
Model: openai-community/gpt2


In [ ]:

# Step 2: Text Generation Function

def continue_text(
    prompt,
    sample=False,
    temperature=0.8,
    seed=42
):

    torch.manual_seed(seed)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    options = {
        "do_sample": sample,
        "max_new_tokens": 45,
        "pad_token_id": tokenizer.eos_token_id
    }

    if sample:
        options.update(
            temperature=temperature,
            top_p=0.9
        )

    with torch.no_grad():
        tokens = model.generate(
            **inputs,
            **options
        )

    return tokenizer.decode(
        tokens[0, inputs.input_ids.size(1):],
        skip_special_tokens=True
    ).strip()


print("Text generation function created!")


In [4]:



prompts = [
    "A story about a robot begins like this:",
    "In a school library, a tiny robot found a lost book. It"
]

for prompt in prompts:

    print("\nPROMPT:", prompt)


    print("GREEDY:", continue_text(prompt))

    for temp in (0.6, 1.0):

        for seed in (42, 43):

            result = continue_text(
                prompt,
                sample=True,
                temperature=temp,
                seed=seed
            )

            print(
                "Temperature:", temp,
                "Seed:", seed,
                "\nOutput:", result
            )



PROMPT: A story about a robot begins like this:
GREEDY: A robot is a robot that is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed to do something. It is programmed
Temperature: 0.6 Seed: 42 
Output: The robot was a humanoid robot with a heart, eyes, and hands that looked like a human, but its skin was red. The robot had a small, muscular body and was very muscular. It was also very slow.
Temperature: 0.6 Seed: 43 
Output: The robot, a robot named "The Robot," is trying to figure out what happened to its owner, who is a woman. The robot tries to figure out how to get back to her, and then it has to figure
Temperature: 1.0 Seed: 42 
Output: The Robot Man was brought to Earth by the Galactic government, to escape the dark and deadly dangers of space. The robot is the most powerful of all the robots, and even better. It has a long memory for what was
Temperature: 1.0 Seed: 43 
Output: The p

In [5]:


examples = [
    (
        "graphs",
        "Sketch a graph, then mark a path through three vertices."
    ),
    (
        "Python loops",
        "Trace a loop by writing the counter after each step."
    ),
    (
        "fractions",
        "Draw equal parts and label the fraction you shaded."
    ),
    (
        "sets",
        "Draw a Venn diagram and label the intersection."
    ),
    (
        "sorting",
        "Sort five values by hand and record each swap."
    ),
    (
        "probability",
        "List the outcomes of two coin tosses before counting."
    )
]

print("Total training examples:", len(examples))

for topic, answer in examples:
    print("\nTopic:", topic)
    print("Study Tip:", answer)


Total training examples: 6

Topic: graphs
Study Tip: Sketch a graph, then mark a path through three vertices.

Topic: Python loops
Study Tip: Trace a loop by writing the counter after each step.

Topic: fractions
Study Tip: Draw equal parts and label the fraction you shaded.

Topic: sets
Study Tip: Draw a Venn diagram and label the intersection.

Topic: sorting
Study Tip: Sort five values by hand and record each swap.

Topic: probability
Study Tip: List the outcomes of two coin tosses before counting.


In [7]:



examples = [
    (
        "graphs",
        "Sketch a graph, then mark a path through three vertices."
    ),
    (
        "Python loops",
        "Trace a loop by writing the counter after each step."
    ),
    (
        "fractions",
        "Draw equal parts and label the fraction you shaded."
    ),
    (
        "sets",
        "Draw a Venn diagram and label the intersection."
    ),
    (
        "sorting",
        "Sort five values by hand and record each swap."
    ),
    (
        "probability",
        "List the outcomes of two coin tosses before counting."
    )
]

print("Total training examples:", len(examples))

for topic, answer in examples:
    print("\nTopic:", topic)
    print("Study Tip:", answer)


Total training examples: 6

Topic: graphs
Study Tip: Sketch a graph, then mark a path through three vertices.

Topic: Python loops
Study Tip: Trace a loop by writing the counter after each step.

Topic: fractions
Study Tip: Draw equal parts and label the fraction you shaded.

Topic: sets
Study Tip: Draw a Venn diagram and label the intersection.

Topic: sorting
Study Tip: Sort five values by hand and record each swap.

Topic: probability
Study Tip: List the outcomes of two coin tosses before counting.


In [8]:



tuned = AutoModelForCausalLM.from_pretrained(
    model_id
).to(device)


tuned.requires_grad_(False)

tuned.transformer.h[-1].requires_grad_(True)


opt = torch.optim.AdamW(
    tuned.transformer.h[-1].parameters(),
    lr=2e-5
)


for epoch in range(3):

    tuned.train()

    for topic, answer in examples:

        prefix = tokenizer.encode(
            f"Topic: {topic}\nStudy tip:"
        )

        target = tokenizer.encode(
            " " + answer
        ) + [tokenizer.eos_token_id]

        ids = torch.tensor(
            [prefix + target],
            device=device
        )

        labels = torch.tensor(
            [[-100] * len(prefix) + target],
            device=device
        )

        opt.zero_grad()

        loss = tuned(
            input_ids=ids,
            labels=labels
        ).loss

        loss.backward()
        opt.step()

    print(
        "Epoch:", epoch + 1,
        "Last example loss:", round(loss.item(), 3)
    )


tuned.eval()

tuned.save_pretrained(
    out / "study_tip_gpt2"
)

tokenizer.save_pretrained(
    out / "study_tip_gpt2"
)

print("Fine-tuning completed successfully!")


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch: 1 Last example loss: 4.498
Epoch: 2 Last example loss: 4.929
Epoch: 3 Last example loss: 4.236
Fine-tuning completed successfully!


In [9]:



held_out = [
    "recursion",
    "matrices",
    "binary search"
]

base_model = model

for topic in held_out:

    prompt = f"Topic: {topic}\nStudy tip:"


    model = base_model

    base_output = continue_text(prompt)


    model = tuned.eval()

    tuned_output = continue_text(prompt)

    print("\nTopic:", topic)

    print("BASE MODEL:")
    print(base_output)

    print("FINE-TUNED MODEL:")
    print(tuned_output)


model = base_model



Topic: recursion
BASE MODEL:
Use the recursion keyword to define a new type of recursion.

Example:

class Recursion { public: Recursion(int i) { return i; } public: Recursion(int i)
FINE-TUNED MODEL:
Use the recursion keyword to define a new recursion.

Example:

import recursion from ' recursion ' ; import recursion.recursion ; import recursion.recursion.recursion.rec

Topic: matrices
BASE MODEL:
Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as
FINE-TUNED MODEL:
Use the matrices to create a matrix of the same size as the input.

The following matrix is a matrix of the same size as the input.

The following matrix is a matrix of the same size as

Topic: binary search
BASE MODEL:
Use the search box to find the search term you want to search for.

Study tip: Use the search box to find the search term you want to search for.

Study tip: Use the search 